# OPRB test benchmark — EasyOCR English (CRAFT + english_g2)

Notebook độc lập cho mô hình OCR tiếp theo: **EasyOCR 1.7.2**, gồm bộ phát hiện chữ **CRAFT** và bộ nhận dạng tiếng Anh pretrained **english_g2**. Đây là một pipeline OCR, tính là **một mô hình** trong báo cáo. Không huấn luyện hoặc fine-tune trên test.

| Thành phần | Cấu hình |
|---|---|
| Dữ liệu | OPRB test: 6.000 ảnh text + 3.000 ảnh nontext |
| Đầu vào mô hình | Chỉ ảnh che khuất `test/text/img/`, `test/nontext/img/` |
| Ground truth | `test/*/gt/`, chỉ dùng để chấm điểm |
| Pipeline | CRAFT + English generation-2 (`english_g2`) |
| Kiểu số | Không yêu cầu lượng tử hóa; ghi dtype thực tế và số tham số của cả hai mạng |
| Thiết bị | Một CUDA GPU nếu có; tự chuyển CPU khi GPU không có hoặc chọn `DEVICE_PREFERENCE = "cpu"` |
| Chỉ số | Cùng CER, WER, bag-of-words token precision/recall/F1 và thời gian/ảnh như các notebook Florence/docTR |
| Tiến độ | Thanh tiến độ dạng text và dòng `PROGRESS` được flush khoảng mỗi phút |
| Kết quả | CSV từng ảnh/theo nhóm, cấu hình JSON, biểu đồ và ZIP riêng |

**Cách chạy trên Kaggle:** tạo notebook riêng, bật Internet và chọn GPU nếu được cấp. Giữ `MAX_SAMPLES_PER_CATEGORY = 5` cho lần đầu (10 ảnh, seed 42 giống các mô hình trước). Chạy từ trên xuống, kiểm tra chất lượng và tốc độ rồi mới chuyển sang `None` nếu đánh giá đủ 9.000 ảnh. Nếu dùng tập con cho bài nộp, cần được giảng viên chấp thuận.

Hai notebook Kaggle có phiên chạy và thư mục `/kaggle/working` độc lập; notebook này tự tải dữ liệu nếu không có sẵn. Không cần dừng phiên Florence đang chạy. Nếu không được cấp thêm GPU, có thể chạy notebook này với CPU; báo cáo phải nêu đúng phần cứng khi so sánh thời gian.

**Lưu và tiếp tục:** vòng lặp ghi checkpoint sau mỗi trang và dừng sau tối đa 10 giờ đánh giá để xuất kết quả. Khi chạy một Saved Version mới, gắn output của phiên trước làm Input, rồi đặt `RESUME_FROM` đến thư mục chứa cả `per_image_results.csv` và `checkpoint_config.json`. Các thiết lập mô hình/chấm điểm/thiết bị phải khớp; số ảnh được chọn có thể tăng từ 5 lên `None`.

EasyOCR không sinh transcript theo giới hạn token như VLM. Số vùng chữ được ghi trong `n_text_regions`; trường giới hạn token không áp dụng. Nếu không nhận được vùng chữ, dự đoán rỗng vẫn được chấm điểm, không bị bỏ qua.

Nguồn: [EasyOCR chính thức](https://github.com/JaidedAI/EasyOCR), [API](https://www.jaided.ai/easyocr/documentation/), [mã nguồn v1.7.2](https://github.com/JaidedAI/EasyOCR/tree/v1.7.2), [OPRB](https://huggingface.co/datasets/kpurkayastha/OPRB).


## 1. Cài thư viện

In [ ]:
%pip -q install "easyocr==1.7.2" "huggingface_hub>=0.34,<1" "jiwer==4.0.0" "pandas>=2.0" matplotlib "Pillow>=10,<13" tqdm

# Dùng cặp torch/torchvision có sẵn trong môi trường Kaggle.
# Nếu pip thay một thư viện đã import, restart kernel rồi chạy từ Mục 2.


## 2. Import và kiểm tra thiết bị

In [ ]:
import gc, hashlib, inspect, json, platform, random, re, shutil, tarfile, time, unicodedata
from collections import Counter
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display
from huggingface_hub import hf_hub_download
from jiwer import cer, wer
from tqdm import tqdm
import matplotlib.pyplot as plt

CUDA_AVAILABLE = torch.cuda.is_available()
print('Torch:', torch.__version__, flush=True)
print('CUDA có sẵn:', CUDA_AVAILABLE, flush=True)
if CUDA_AVAILABLE:
    print('GPU đầu tiên:', torch.cuda.get_device_name(0), flush=True)
else:
    print('Sẽ chạy OCR trên CPU. Có thể bật GPU trong Kaggle nếu còn tài nguyên.', flush=True)


## 3. Cấu hình — thử 10 ảnh trước

In [ ]:
REPO_ID = "kpurkayastha/OPRB"
MODEL_ID = "JaidedAI/EasyOCR:craft+english_g2"
DATA_ROOT = Path("/kaggle/working/oprb_test")
MODEL_DIR = Path("/kaggle/working/easyocr_models")

DEVICE_PREFERENCE = "auto"  # "auto", "cuda", hoặc "cpu".
assert DEVICE_PREFERENCE in {"auto", "cuda", "cpu"}
if DEVICE_PREFERENCE == "cuda" and not CUDA_AVAILABLE:
    raise RuntimeError('Đã chọn cuda nhưng không có GPU. Bật GPU hoặc dùng auto/cpu.')
USE_CUDA = CUDA_AVAILABLE and DEVICE_PREFERENCE != "cpu"
DEVICE = "cuda:0" if USE_CUDA else "cpu"
DEVICE_NAME = torch.cuda.get_device_name(0) if USE_CUDA else "CPU"
OUTPUT_DIR = Path(f"/kaggle/working/oprb_easyocr_en_{'gpu' if USE_CUDA else 'cpu'}_results")

# Khi tiếp tục từ output phiên Kaggle trước, gắn output đó làm Input rồi điền:
# RESUME_FROM = Path("/kaggle/input/.../oprb_easyocr_en_gpu_results")
RESUME_FROM = None

CATEGORIES = ["text", "nontext"]
MAX_SAMPLES_PER_CATEGORY = 5  # None để chọn toàn bộ 9.000 ảnh.
RANDOM_SEED = 42
NORMALIZE_CASE = True
MAX_EVAL_HOURS = 10.0

LANGUAGES = ["en"]
DETECT_NETWORK = "craft"
RECOG_NETWORK = "english_g2"
GPU_RECOGNITION_BATCH_SIZE = 16
# EasyOCR nhận dạng từng crop trên CPU; không báo batch GPU như batch CPU.
OCR_BATCH_SIZE = GPU_RECOGNITION_BATCH_SIZE if USE_CUDA else 1
READTEXT_KWARGS = {
    'decoder': 'greedy',
    'batch_size': OCR_BATCH_SIZE,
    'workers': 0,
    'detail': 1,
    'paragraph': False,
    'canvas_size': 2560,
    'mag_ratio': 1.0,
    'min_size': 20,
    'text_threshold': 0.7,
    'low_text': 0.4,
    'link_threshold': 0.4,
    'rotation_info': None,
    'output_format': 'standard',
}

assert MAX_EVAL_HOURS is None or MAX_EVAL_HOURS > 0
assert MAX_SAMPLES_PER_CATEGORY is None or MAX_SAMPLES_PER_CATEGORY > 0
assert OCR_BATCH_SIZE > 0
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('Pipeline:', MODEL_ID, flush=True)
print('Thiết bị:', DEVICE, DEVICE_NAME, flush=True)
print('Mỗi loại:', MAX_SAMPLES_PER_CATEGORY, '; seed:', RANDOM_SEED, flush=True)
print('Output:', OUTPUT_DIR, flush=True)


## 4. Tải ảnh test và GT dùng để chấm điểm

Giữ nguyên cách tải/extract của notebook trước. Chỉ tải `test/{text,nontext}/img.tar.gz` và `gt.tar.gz`. Không tải train, `img_gt/` hoặc `annots/`. Nếu có đủ dữ liệu ở `DATA_ROOT` thì tái sử dụng; chọn 10 ảnh vẫn cần tải toàn bộ các archive tương ứng.


In [ ]:
def extract_flat(archive_path: str, destination: Path, allowed_suffix: str) -> int:
    """Safely copy only files of the specified type; never use archive-supplied paths."""
    destination.mkdir(parents=True, exist_ok=True)
    count = 0
    with tarfile.open(archive_path, mode="r:gz") as archive:
        for member in tqdm(archive, desc=f"Extract {destination.parent.name}/{destination.name}", leave=False):
            if not member.isfile() or not member.name.lower().endswith(allowed_suffix):
                continue
            # Ignore any member directory traversal; write only basenames.
            filename = Path(member.name).name
            target = destination / filename
            if not target.exists():
                stream = archive.extractfile(member)
                if stream is None:
                    continue
                with stream, target.open("wb") as out:
                    shutil.copyfileobj(stream, out)
            count += 1
    return count


def download_test_category(category: str):
    if category not in {"text", "nontext"}:
        raise ValueError(f"Unknown category: {category}")
    for kind, suffix in (("img", ".jpg"), ("gt", ".txt")):
        repo_path = f"test/{category}/{kind}.tar.gz"
        target_dir = DATA_ROOT / category / kind
        existing = list(target_dir.glob(f"*{suffix}")) if target_dir.exists() else []
        expected = 6000 if category == "text" else 3000
        if len(existing) >= expected:
            print(f"Reuse {category}/{kind}: {len(existing)} files")
            continue
        print(f"Downloading {repo_path} ...")
        tar_path = hf_hub_download(repo_id=REPO_ID, repo_type="dataset", filename=repo_path)
        n = extract_flat(tar_path, target_dir, suffix)
        print(f"  {category}/{kind}: {n:,} archive members of type {suffix} -> {target_dir}")


for category in CATEGORIES:
    download_test_category(category)

## 5. Ghép ảnh và GT — cùng danh sách, seed như Florence/docTR

In [ ]:
def parse_occlusion_metadata(stem: str):
    # OPRB filename convention: OcclusionType__density__batchID... 
    parts = stem.split("__", 2)
    return {
        "occlusion_type": parts[0] if parts else "unknown",
        "occlusion_density": parts[1] if len(parts) > 1 else "unknown",
    }


def collect_test_pairs(categories):
    pairs = []
    for category in categories:
        image_dir = DATA_ROOT / category / "img"
        gt_dir = DATA_ROOT / category / "gt"
        images = {p.stem: p for p in image_dir.glob("*.jpg")}
        gts = {p.stem: p for p in gt_dir.glob("*.txt")}
        common = sorted(images.keys() & gts.keys())
        print(f"{category}: images={len(images):,}, gt={len(gts):,}, paired={len(common):,}")
        if not common:
            raise RuntimeError(f"No paired files for {category}. Check extraction and naming.")
        if MAX_SAMPLES_PER_CATEGORY is not None:
            rng = random.Random(RANDOM_SEED)
            common = sorted(rng.sample(common, min(MAX_SAMPLES_PER_CATEGORY, len(common))))
        for stem in common:
            pairs.append({
                "category": category,
                "sample_id": stem,
                "image_path": str(images[stem]),
                "gt_path": str(gts[stem]),
                **parse_occlusion_metadata(stem),
            })
    return pairs


pairs = collect_test_pairs(CATEGORIES)
print(f"Selected {len(pairs):,} samples. Set MAX_SAMPLES_PER_CATEGORY=None to use the entire test set.")
display(pd.DataFrame(pairs)[["category", "occlusion_type", "occlusion_density", "sample_id"]].head(8))

with Image.open(pairs[0]["image_path"]) as image:
    preview = image.copy()
preview.thumbnail((650, 850))
display(preview)

## 6. Đọc GT và tính điểm

Giữ nguyên hàm đọc GT, chuẩn hóa và tính điểm từ notebook trước. GT OPRB có hàng DocBank 10 cột; hàm đọc cũng chấp nhận hàng 11 cột. Chỉ lấy token ở cột đầu, giữ thứ tự dòng, ghép bằng dấu cách. GT rỗng báo lỗi trước suy luận.

- CER/WER: khoảng cách chỉnh sửa chia độ dài tham chiếu, càng thấp càng tốt; có thể vượt 1.0.
- Token P/R/F1: giao đa tập token có xét số lần xuất hiện, không xét thứ tự; không phải F1 hộp phát hiện chữ.
- Chuẩn hóa: Unicode NFKC, khoảng trắng, casefold khi `NORMALIZE_CASE=True`.
- Kết quả tổng hợp là macro theo từng ảnh. Dự đoán OCR rỗng vẫn được chấm điểm.


In [ ]:
def read_gt_tokens(gt_path: str):
    tokens = []
    with open(gt_path, "r", encoding="utf-8-sig", errors="replace") as file:
        for line in file:
            fields = line.rstrip("\r\n").split("\t")
            if len(fields) >= 10 and fields[0].strip():
                tokens.append(fields[0].strip())
    if not tokens:
        raise ValueError(f'No GT tokens read from {gt_path}. Check the annotation format and file contents.')
    return tokens


def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.casefold() if NORMALIZE_CASE else text


def bag_of_words_prf(reference: str, prediction: str):
    ref = Counter(reference.split())
    hyp = Counter(prediction.split())
    matched = sum((ref & hyp).values())
    ref_n, hyp_n = sum(ref.values()), sum(hyp.values())
    precision = matched / hyp_n if hyp_n else (1.0 if ref_n == 0 else 0.0)
    recall = matched / ref_n if ref_n else (1.0 if hyp_n == 0 else 0.0)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


def compute_metrics(ground_truth: str, prediction: str):
    ref, hyp = clean_text(ground_truth), clean_text(prediction)
    if not ref:
        raise ValueError("Empty GT transcript; cannot compute meaningful CER/WER.")
    p, r, f1 = bag_of_words_prf(ref, hyp)
    return {
        "cer": float(cer(ref, hyp)),
        "wer": float(wer(ref, hyp)),
        "token_precision": p,
        "token_recall": r,
        "token_f1": f1,
        "gt_words": len(ref.split()),
        "pred_words": len(hyp.split()),
    }


# Check metric code on a toy example BEFORE downloading the model.
assert compute_metrics("The cat sat", "The cat")['wer'] == 1/3
print('Toy check:', compute_metrics("The cat sat", "The cat"))
print('First GT tokens:', read_gt_tokens(pairs[0]['gt_path'])[:35])

## 7. Tải CRAFT và English generation-2 pretrained

EasyOCR tự tải hai checkpoint chính thức vào `MODEL_DIR` và kiểm tra MD5. Notebook ghi thêm SHA-256, dung lượng, dtype và số tham số của từng mạng để mô tả đầy đủ pipeline trong báo cáo. Không yêu cầu dynamic quantization. Nếu EasyOCR bọc model bằng DataParallel, notebook bỏ lớp bọc để chạy trên đúng một GPU, tương ứng cách dùng GPU của Florence.


In [ ]:
import easyocr
from easyocr.config import detection_models, recognition_models

reader = easyocr.Reader(
    LANGUAGES,
    gpu=DEVICE if USE_CUDA else False,
    model_storage_directory=str(MODEL_DIR),
    user_network_directory=str(MODEL_DIR / 'user_network'),
    detect_network=DETECT_NETWORK,
    recog_network=RECOG_NETWORK,
    download_enabled=True,
    quantize=False,
    cudnn_benchmark=False,
    verbose=True,
)

# Cố định một GPU cho cả detector và recognizer, không gộp hai GPU ngầm.
for component_name in ('detector', 'recognizer'):
    network = getattr(reader, component_name)
    if isinstance(network, torch.nn.DataParallel):
        network = network.module.to(DEVICE)
        setattr(reader, component_name, network)
    network.eval()


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


weight_specs = [
    ('detector', detection_models[DETECT_NETWORK]),
    ('recognizer', recognition_models['gen2'][RECOG_NETWORK]),
]
MODEL_COMPONENTS = []
for component_name, specification in weight_specs:
    path = MODEL_DIR / specification['filename']
    network = getattr(reader, component_name)
    MODEL_COMPONENTS.append({
        'component': component_name,
        'checkpoint_filename': path.name,
        'checkpoint_url': specification['url'],
        'sha256': sha256_file(path),
        'weight_file_bytes': path.stat().st_size,
        'parameter_count': int(sum(parameter.numel() for parameter in network.parameters())),
        'parameter_dtypes': sorted({str(parameter.dtype) for parameter in network.parameters()}),
        'quantized_module_types': sorted({
            type(module).__module__ + '.' + type(module).__name__
            for module in network.modules() if 'quantized' in type(module).__module__
        }),
    })

# Lưu cả các mặc định chưa ghi đè của đúng bản EasyOCR đã cài.
EFFECTIVE_READTEXT_SETTINGS = {
    name: parameter.default
    for name, parameter in inspect.signature(reader.readtext).parameters.items()
    if parameter.default is not inspect.Parameter.empty
}
EFFECTIVE_READTEXT_SETTINGS.update(READTEXT_KWARGS)
print(json.dumps(MODEL_COMPONENTS, indent=2), flush=True)
print('Tổng số tham số:', sum(component['parameter_count'] for component in MODEL_COMPONENTS), flush=True)
print('Thiết bị:', DEVICE_NAME, '; batch crop:', OCR_BATCH_SIZE, flush=True)


## 8. OCR một trang và warm-up

Mô hình chỉ nhận đường dẫn ảnh che khuất. EasyOCR tự đọc ảnh, phát hiện vùng chữ và nhận dạng. Ghép văn bản theo thứ tự các vùng mà EasyOCR trả về, không sắp theo GT, không lọc văn bản theo GT hoặc confidence. Thứ tự này có thể chưa phù hợp với trang nhiều cột.

Không có prompt hay giới hạn sinh token. Ghi số vùng trả về; không nhận ra chữ vẫn là kết quả dự đoán rỗng hợp lệ. Trang ví dụ này warm-up trước khi đo các ảnh ở Mục 9.


In [ ]:
@torch.inference_mode()
def transcribe_page(image_path: str):
    """Chỉ nhận ảnh che khuất; không nhận GT, token hoặc hộp GT."""
    if USE_CUDA:
        torch.cuda.synchronize()
    started = time.perf_counter()
    regions = reader.readtext(str(image_path), **READTEXT_KWARGS)
    pieces = []
    for _box, text, _confidence in regions:
        text = text.strip()
        if text:
            pieces.append(text)
    prediction = ' '.join(pieces)
    if USE_CUDA:
        torch.cuda.synchronize()
    elapsed_sec = time.perf_counter() - started
    return prediction, elapsed_sec, int(len(regions))


sample = pairs[0]
example_gt = ' '.join(read_gt_tokens(sample['gt_path']))
example_prediction, example_time, example_regions = transcribe_page(sample['image_path'])
print('CATEGORY:', sample['category'], flush=True)
print('IMAGE:', Path(sample['image_path']).name, flush=True)
print('LATENCY (s):', round(example_time, 2), flush=True)
print('OCR REGIONS:', example_regions, flush=True)
print('GT (first 700 chars):', example_gt[:700], flush=True)
print('PREDICTION (first 700 chars):', example_prediction[:700], flush=True)
print('METRICS:', compute_metrics(example_gt, example_prediction), flush=True)


## 9. Đánh giá, log tiến độ và checkpoint

Checkpoint được ghi nguyên tử sau mỗi trang. Chỉ tái sử dụng ảnh đã thành công; ảnh lỗi được thử lại ở lần tiếp theo. Mỗi trang được nhận diện bằng `(category, sample_id)`, tránh trùng ID giữa text/nontext. Có thể tăng số ảnh từ 5 lên `None` mà giữ các dự đoán đã có.

Mỗi lần chạy in `PROGRESS` sau ảnh đầu tiên, khoảng mỗi 60 giây và sau ảnh cuối. Dòng log có số ảnh thành công, lỗi, thời gian và ETA cho ảnh chưa thử. Trong Saved Version của Kaggle, theo dõi **View Logs**. Tối đa 10 giờ đánh giá rồi vòng lặp dừng để Mục 10–12 xuất phần kết quả đã có; nếu còn ảnh thì cần tiếp tục sau.

Một Saved Version mới không tự nhận checkpoint của phiên tương tác. Gắn output phiên trước làm Input, đặt `RESUME_FROM` vào đúng thư mục CSV/JSON; notebook phục hồi hai tệp khi thư mục output hiện tại chưa có CSV. Không trộn checkpoint CPU với GPU hoặc thay đổi cấu hình trên cùng thư mục kết quả.


In [ ]:
RESULTS_CSV = OUTPUT_DIR / 'per_image_results.csv'
CHECKPOINT_CONFIG = OUTPUT_DIR / 'checkpoint_config.json'
checkpoint_settings = {
    'dataset': REPO_ID,
    'split': 'test',
    'model': MODEL_ID,
    'languages': LANGUAGES,
    'detector': DETECT_NETWORK,
    'recognizer': RECOG_NETWORK,
    'components': MODEL_COMPONENTS,
    'dynamic_quantization_requested': False,
    'readtext_settings': EFFECTIVE_READTEXT_SETTINGS,
    'reading_order': 'EasyOCR returned region order; join with spaces',
    'normalize_case': NORMALIZE_CASE,
    'metric_definition': 'Macro means of per-image CER/WER and bag-of-words overlap',
    'device': DEVICE,
    'device_name': DEVICE_NAME,
    'torch_num_threads': torch.get_num_threads(),
    'easyocr_version': version('easyocr'),
    'torch_version': str(torch.__version__),
    'torchvision_version': version('torchvision'),
}

if RESUME_FROM is not None and not RESULTS_CSV.exists():
    resume_root = Path(RESUME_FROM)
    for filename in ('checkpoint_config.json', 'per_image_results.csv'):
        if not (resume_root / filename).is_file():
            raise FileNotFoundError(f'Missing resume file: {resume_root / filename}')
    for filename in ('checkpoint_config.json', 'per_image_results.csv'):
        source = resume_root / filename
        target = OUTPUT_DIR / filename
        if target.exists() and target.read_bytes() != source.read_bytes():
            raise FileExistsError(f'Không ghi đè checkpoint khác: {target}')
        if source.resolve() != target.resolve():
            shutil.copy2(source, target)
    print('Restored checkpoint from:', resume_root, flush=True)

if CHECKPOINT_CONFIG.exists():
    saved_settings = json.loads(CHECKPOINT_CONFIG.read_text(encoding='utf-8'))
    if saved_settings != checkpoint_settings:
        raise ValueError('Checkpoint khác mô hình/cấu hình/thiết bị. Chọn OUTPUT_DIR mới cho thí nghiệm khác.')
elif RESULTS_CSV.exists():
    raise ValueError('CSV thiếu checkpoint_config.json. Phục hồi cả hai tệp hoặc chọn OUTPUT_DIR mới.')
else:
    CHECKPOINT_CONFIG.write_text(json.dumps(checkpoint_settings, indent=2), encoding='utf-8')


def sample_key(record):
    return record['category'], record['sample_id']


selected_keys = {sample_key(sample) for sample in pairs}
if RESULTS_CSV.exists():
    previous = pd.read_csv(RESULTS_CSV, keep_default_na=False).to_dict(orient='records')
    if any(record.get('model_id') != MODEL_ID for record in previous):
        raise ValueError('CSV chứa mô hình khác. Chọn OUTPUT_DIR riêng.')
    saved_rows = {sample_key(record): record for record in previous}
else:
    saved_rows = {}
finished = {key for key, record in saved_rows.items() if record.get('status') == 'ok'}
pending = [sample for sample in pairs if sample_key(sample) not in finished]
print(f'Reusing {len(finished & selected_keys):,} successful selected pages; {len(pending):,} remaining.', flush=True)
evaluation_started = time.perf_counter()
last_progress_log = evaluation_started
successful_count = len(finished & selected_keys)
errors_this_run = 0


def format_duration(seconds):
    hours, remainder = divmod(int(seconds), 3600)
    minutes, seconds = divmod(remainder, 60)
    return f'{hours:02d}:{minutes:02d}:{seconds:02d}'


for attempted_this_run, sample in enumerate(tqdm(pending, desc='EasyOCR evaluation', unit='page', mininterval=60), start=1):
    if MAX_EVAL_HOURS is not None and time.perf_counter() - evaluation_started >= MAX_EVAL_HOURS * 3600:
        print('Time budget reached. Export this checkpoint, then resume later.', flush=True)
        break
    result = dict(sample)
    result['model_id'] = MODEL_ID
    try:
        ground_truth = ' '.join(read_gt_tokens(sample['gt_path']))
        prediction, elapsed, n_regions = transcribe_page(sample['image_path'])
        result.update({
            'status': 'ok',
            'prediction': prediction,
            'ground_truth': ground_truth,
            'latency_sec': elapsed,
            'n_text_regions': n_regions,
            **compute_metrics(ground_truth, prediction),
        })
    except Exception as exc:
        result.update({'status': 'error', 'error': repr(exc)})
        print('Failure:', sample['sample_id'], repr(exc), flush=True)
        gc.collect()
        if USE_CUDA:
            torch.cuda.empty_cache()
    saved_rows[sample_key(sample)] = result
    temporary_csv = RESULTS_CSV.with_suffix('.csv.tmp')
    pd.DataFrame(saved_rows.values()).to_csv(temporary_csv, index=False)
    temporary_csv.replace(RESULTS_CSV)
    successful_count += result['status'] == 'ok'
    errors_this_run += result['status'] == 'error'
    now = time.perf_counter()
    if attempted_this_run == 1 or now - last_progress_log >= 60 or attempted_this_run == len(pending):
        elapsed = now - evaluation_started
        average_sec = elapsed / attempted_this_run
        unattempted = len(pending) - attempted_this_run
        print(
            f'PROGRESS: {successful_count:,}/{len(pairs):,} successful ({100 * successful_count / len(pairs):.1f}%) | '
            f'attempted this run={attempted_this_run:,}/{len(pending):,} | errors this run={errors_this_run:,} | '
            f'elapsed={format_duration(elapsed)} | avg={average_sec:.2f}s/page | '
            f'ETA for unattempted pages={format_duration(unattempted * average_sec)}',
            flush=True,
        )
        last_progress_log = now

rows = [record for key, record in saved_rows.items() if key in selected_keys]
results = pd.DataFrame(rows)
if results.empty:
    raise RuntimeError('Chưa có ảnh được thử. Kiểm tra giới hạn thời gian và checkpoint.')
print('Saved:', RESULTS_CSV, flush=True)
print('Successful:', (results.status == 'ok').sum(), '/', len(pairs), 'selected pages', flush=True)
display(results.reindex(columns=[
    'category', 'occlusion_type', 'cer', 'wer', 'token_f1',
    'latency_sec', 'n_text_regions', 'status', 'error'
]).head(10))


## 10. Tổng hợp chỉ số và thời gian

In [ ]:
ok = results.loc[results['status'] == 'ok'].copy()
assert len(ok), 'Không có trang chấm điểm được. Kiểm tra cột error.'
metric_cols = ['cer', 'wer', 'token_precision', 'token_recall', 'token_f1', 'latency_sec']
# CSV có hàng lỗi có thể khiến pandas đọc các số thành chuỗi khi resume.
for column in metric_cols + ['n_text_regions']:
    ok[column] = pd.to_numeric(ok[column], errors='raise')


def aggregate_metrics(df, group_by):
    return df.groupby(group_by, dropna=False).agg(
        n=('sample_id', 'count'),
        **{f'mean_{name}': (name, 'mean') for name in metric_cols},
        mean_text_regions=('n_text_regions', 'mean'),
    ).reset_index()


overall = {f'mean_{name}': float(ok[name].mean()) for name in metric_cols}
overall.update({
    'n_selected': int(len(pairs)),
    'n_success': int(len(ok)),
    'n_errors': int(len(results) - len(ok)),
    'n_remaining': int(len(pairs) - len(ok)),
    'evaluation_complete': bool(len(ok) == len(pairs)),
    'n_empty_predictions': int(ok['prediction'].fillna('').astype(str).str.strip().eq('').sum()),
    'mean_text_regions': float(ok['n_text_regions'].mean()),
})
by_category = aggregate_metrics(ok, ['category'])
by_type = aggregate_metrics(ok, ['category', 'occlusion_type'])
by_density = aggregate_metrics(ok, ['category', 'occlusion_density'])
print('OVERALL macro-averaged metrics:', flush=True)
print(json.dumps(overall, indent=2), flush=True)
print('BY OCCLUSION REGION:', flush=True)
display(by_category)
print('BY OCCLUSION TYPE:', flush=True)
display(by_type)
print('BY OCCLUSION DENSITY:', flush=True)
display(by_density)
by_category.to_csv(OUTPUT_DIR / 'summary_by_category.csv', index=False)
by_type.to_csv(OUTPUT_DIR / 'summary_by_occlusion_type.csv', index=False)
by_density.to_csv(OUTPUT_DIR / 'summary_by_density.csv', index=False)

region_times = by_category.set_index('category')['mean_latency_sec']
if {'text', 'nontext'}.issubset(region_times.index):
    estimated_full_hours = (6000 * region_times['text'] + 3000 * region_times['nontext']) / 3600
    print(f'Ước lượng suy luận 9.000 trang trên {DEVICE_NAME}: {estimated_full_hours:.1f} giờ.', flush=True)
print('Số ảnh còn cần chấm thành công:', overall['n_remaining'], flush=True)


## 11. Biểu đồ kết quả

In [ ]:
ax = by_category.set_index('category')[['mean_cer', 'mean_wer']].plot(
    kind='bar', figsize=(7, 4), rot=0, title='OPRB test: mean CER and WER by region'
)
ax.set_xlabel('Occlusion region')
ax.set_ylabel('Error rate (lower is better)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cer_wer_by_category.png', dpi=150)
plt.show()

ax = by_type.set_index('occlusion_type').groupby(level=0)['mean_wer'].mean().sort_values().plot(
    kind='barh', figsize=(9, 5), title='Mean WER by occlusion type (unweighted category average)'
)
ax.set_xlabel('Mean word error rate (lower is better)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'wer_by_occlusion_type.png', dpi=150)
plt.show()

## 12. Xuất ZIP và cấu hình tái hiện

In [ ]:
run_config = {
    **checkpoint_settings,
    'categories': CATEGORIES,
    'no_finetuning': True,
    'max_samples_per_category': MAX_SAMPLES_PER_CATEGORY,
    'random_seed': RANDOM_SEED,
    'max_eval_hours_per_run': MAX_EVAL_HOURS,
    'metrics': overall,
    'gpu_name': DEVICE_NAME if USE_CUDA else None,
    'cuda_version': torch.version.cuda if USE_CUDA else None,
    'platform': platform.platform(),
    'jiwer_version': version('jiwer'),
    'pandas_version': version('pandas'),
    'numpy_version': np.__version__,
    'output_token_limit': None,
    'output_token_limit_note': 'Not applicable to this detection + CTC recognition pipeline.',
}
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
zip_name = shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR)
print('Results ZIP:', zip_name, flush=True)
print('Tải ZIP trong Output của Kaggle.', flush=True)
print('Lần sau: gắn output phiên này làm Input rồi đặt RESUME_FROM đến thư mục CSV/JSON.', flush=True)
if not overall['evaluation_complete']:
    print('Đây là kết quả một phần. Tiếp tục checkpoint để hoàn tất tập đã chọn.', flush=True)


## Ghi báo cáo và giới hạn

Notebook này bổ sung **một pipeline OCR** cho yêu cầu ít nhất năm mô hình. Báo cáo PDF dưới bốn trang A4 cần: Giới thiệu, Thống kê dữ liệu, Mô tả mô hình, Kết quả thực nghiệm, Kết luận.

- Ghi rõ detector CRAFT, recognizer `english_g2`, phiên bản EasyOCR, checkpoint/hash, số tham số và dung lượng của **cả hai mạng**. Thông tin này có trong `components` của `run_config.json`.
- Báo cáo đúng thiết bị CPU/GPU và kiểu số thực tế. Không diễn giải chênh lệch thời gian CPU/GPU như so sánh trên cùng phần cứng. Thời gian bắt đầu sau warm-up và không gồm tải trọng số/dữ liệu, chấm điểm hoặc ghi checkpoint.
- Giữ cùng ảnh, GT, chuẩn hóa và metric giữa các mô hình. Các tỷ lệ CER/WER/P/R/F1 trong CSV cần nhân 100 nếu bảng báo cáo dùng %. Đó là macro theo ảnh, không phải corpus CER/WER.
- Bộ tiếng Anh có giới hạn bảng ký tự và thường khó với công thức, chữ Hy Lạp, bảng và layout nhiều cột. Ghép vùng theo thứ tự EasyOCR trả về có thể khác thứ tự GT. Không dùng GT để sửa thứ tự hoặc bổ sung văn bản dự đoán.
- GT của trang sạch có cả chữ bị che; điều này có thể tăng lỗi. Chỉ đánh giá page transcription, không phải phục hồi ảnh hoặc phát hiện hộp GT.
- Không có giới hạn sinh token của VLM; trường đó là không áp dụng. Một transcript thiếu chữ có thể do phát hiện/nhận dạng sai, không vì thế được bỏ qua. Dự đoán rỗng được chấm điểm và đếm trong `n_empty_predictions`.
- 10 ảnh đầu là thử nghiệm. `evaluation_complete=True` chỉ nói đã hoàn tất số ảnh đang chọn; đối chiếu cả `n_selected`, `n_success`, `n_remaining` trước khi ghi hoàn tất 9.000 ảnh. Sampling cho bài nộp cần đúng yêu cầu giảng viên.
- Chọn output riêng nếu đổi mô hình, thuật toán OCR, chấm điểm hoặc thiết bị. Thành công được tái sử dụng; lỗi được thử lại. Saved Version mới cần phục hồi checkpoint rõ ràng qua `RESUME_FROM`.

Nguồn: [EasyOCR](https://github.com/JaidedAI/EasyOCR), [API](https://www.jaided.ai/easyocr/documentation/), [nguồn v1.7.2 và cấu hình weights](https://github.com/JaidedAI/EasyOCR/tree/v1.7.2/easyocr), [OPRB](https://huggingface.co/datasets/kpurkayastha/OPRB).
